In [1]:
import pandas as pd
import numpy as np
import sys
import webbrowser, os
import json
import boto3
import re
import sagemaker
import time
import uuid
from sagemaker import get_execution_role
from sklearn.model_selection import train_test_split
from pprint import pprint

## Convolutional Neural Network for text classification

### Preparing our dataset

In [2]:
df = pd.read_json('./startrek/raw/all_scripts_raw.json')
df.head()

,DS9,TOS,TAS,TNG,VOY,ENT
episode 0,\n\n\n\n\n\nThe Deep Space Nine Transcripts - ...,\n\n\n\n\n\nThe Star Trek Transcripts - The Ca...,\n\n\n\n\n\nThe Animated Star Trek Transcripts...,\n\n\n\n\n\nThe Next Generation Transcripts - ...,\n\n\n\n\nThe Voyager Transcripts - Caretaker\...,\n\n\n\n\n\nThe Enterprise Transcripts - Broke...
episode 1,\n\n\n\n\n\nThe Deep Space Nine Transcripts - ...,\n\n\n\n\n\nThe Star Trek Transcripts - The Ma...,\n\n\n\n\n\nThe Animated Star Trek Transcripts...,\n\n\n\n\n\nThe Next Generation Transcripts - ...,\n\n\n\n\n\nThe Voyager Transcripts - Parallax...,\n\n\n\n\n\nThe Enterprise Transcripts - Fight...
episode 2,\n\n\n\n\n\nThe Deep Space Nine Transcripts - ...,\n\n\n\n\n\nThe Star Trek Transcripts - Charli...,\n\n\n\n\n\nThe Animated Star Trek Transcripts...,\n\n\n\n\n\nThe Next Generation Transcripts - ...,\n\n\n\n\n\nThe Voyager Transcripts - Time and...,\n\n\n\n\n\nThe Enterprise Transcripts - Stran...
episode 3,\n\n\n\n\n\nThe Deep Space Nine Transcripts - ...,\n\n\n\n\n\nThe Star Trek Transcripts - Where ...,\n\n\n\n\n\nThe Animated Star Trek Transcripts...,\n\n\n\n\n\nThe Next Generation Transcripts - ...,\n\n\n\n\n\n\nThe Voyager Transcripts - Phage\...,\n\n\n\n\n\nThe Enterprise Transcripts - Unexp...
episode 4,\n\n\n\n\n\nThe Deep Space Nine Transcripts - ...,\n\n\n\n\n\nThe Star Trek Transcripts - The Na...,\n\n\n\n\n\nThe Animated Star Trek Transcripts...,\n\n\n\n\n\nThe Next Generation Transcripts - ...,\n\n\n\n\n\nThe Voyager Transcripts - The Clou...,\n\n\n\n\n\nThe Enterprise Transcripts - Terra...


In [3]:
df.shape

(176, 6)

In [4]:
# Get the script of the first 40 episodes of StarTrek The Next Generation and remove all commentary and retain only the first line of character dialogues
transcript_full = ''
for n in range(40):
    transcript_full += df['TNG'].values[n]
transcript_sentences = transcript_full.split('\n')
clean_sentences = []
for sentence in transcript_sentences:
    if ':' in sentence and 'date' not in sentence:
        clean_sentences.append(sentence)
print(clean_sentences[:10])

["PICARD: You will agree, Data, that Starfleet's", 'DATA: Difficult? Simply solve the mystery of Farpoint Station. ', 'PICARD: As simple as that. ', 'TROI: Farpoint Station. Even the name sounds mysterious. ', "PICARD: It's hardly simple, Data, to negotiate a friendly agreement for", 'DATA: Inquiry. The word snoop? ', 'PICARD: Data, how can you be programmed as a virtual encyclopedia of', 'DATA: Possibility, a kind of human behaviour I was not designed to', 'PICARD: It means to spy, to sneak. ', 'DATA: Ah! To seek covertly, to go stealthily, to slink, slither ']


In [5]:
# Now the character will be the label and their dialogue the text sentence for our classification model
# define a dataframe for the episode
episode_df = pd.DataFrame(columns=['character','dialogue'])
i = 0
for dialogue in clean_sentences:
    episode_df.at[i,'character'] = dialogue.split(':')[0].split('[')[0]
    episode_df.at[i,'dialogue'] = dialogue.split(':')[1]
    i += 1
episode_df.head()

,character,dialogue
0,PICARD,"You will agree, Data, that Starfleet's"
1,DATA,Difficult? Simply solve the mystery of Farpoi...
2,PICARD,As simple as that.
3,TROI,Farpoint Station. Even the name sounds myster...
4,PICARD,"It's hardly simple, Data, to negotiate a frie..."


In [6]:
episode_df.shape

(15413, 2)

### Tokenizing our dataset

In [ ]:
# Install the Spacy library
!pip install -U spacy --quiet

#### Load Spacy model

In [7]:
# Download the English medium size model for our needs
!python -m spacy download en_core_web_lg --quiet

You should consider upgrading via the '/usr/local/bin/python -m pip install --upgrade pip' command.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [8]:
import spacy
spacy_model = spacy.load('en_core_web_lg')
# Print the pre-added list of tasks the model performs on your data
# This is good to know currently, but we will use this later when we run text classification using Spacy
spacy_model.pipe_names

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

#### The snippet below is compute intensive and may run for a minute or two

In [9]:
# Lets get Spacy's interpretation of the dialogue and clean it up
for idx, row in episode_df.iterrows():
    spacy_dialogue = spacy_model(row['dialogue'])
    # remove stop words, convert to lowercase
    dialogue_punct_removed = ''
    for token in spacy_model(spacy_dialogue):
        if token.is_punct==False:
            dialogue_punct_removed += str(token) + ' '
    episode_df.at[idx, 'dialogue'] = dialogue_punct_removed.lower()
    # convert character names to lowercase
    episode_df.at[idx, 'character'] = row['character'].lower()

In [10]:
episode_df.head()

,character,dialogue
0,picard,you will agree data that starfleet 's
1,data,difficult simply solve the mystery of farpoi...
2,picard,as simple as that
3,troi,farpoint station even the name sounds myster...
4,picard,it 's hardly simple data to negotiate a frie...


In [11]:
episode_df['character'].value_counts()

picard      3402
riker       1987
data        1530
laforge      853
worf         652
            ... 
toya           1
alien 2        1
korris         1
t'su           1
alien 1        1
Name: character, Length: 312, dtype: int64

In [12]:
# we will just use dialogues from main characters
epi_new_df = episode_df.query("character in ('picard','riker','data')")

#### Determine unique words to get our vocabulary 

In [13]:
all_unique_words = []
word_list = []
epi_sentences = epi_new_df['dialogue'].values.tolist()
for sentence in epi_sentences:
    for word in sentence.split(' '):
        if word != '':
            word_list.append(word)

all_unique_words = list(set(word_list))

In [14]:
len(all_unique_words)

4805

In [15]:
big_vocab = all_unique_words[:250]
small_vocab = all_unique_words[:50]

In [16]:
# Use the SciKit learn library to split our dataset into train and test dataset - 85% for training and 15% for testing
# Remember that the character is the label we want to predict from the dialogue which is our input text
dialogue_train, dialogue_test, character_train, character_test = train_test_split(epi_new_df['dialogue'],epi_new_df['character'], test_size=0.15)

In [ ]:
# use Keras to get encoding for our dialogues
from keras.preprocessing.text import one_hot
big_vocab_len = len(big_vocab)
dialogue_token_train = [one_hot(words, big_vocab_len) for words in dialogue_train] 
dialogue_token_test = [one_hot(words, big_vocab_len) for words in dialogue_test]

#### Now that we have obtained the encoded integers for our dialogues, let us create a dictionary for our characters for easy translation

In [ ]:
# We will use the GENSIM library for this
!pip install -U gensim --quiet

In [ ]:
# Now create a Gensim dictionary object
from gensim import corpora
temp_list = []
new_list = []
#dedupe away
temp_list = [*set(epi_new_df['character'].values.tolist())]
#strip away
for char in temp_list:
    new_list.append(char.strip(' ').strip(', ').strip(' + ').strip("'"))
new_list
gensim_dict = corpora.Dictionary([new_list])
# get token IDs
char_dict = {}
char_dict = gensim_dict.token2id

In [ ]:
character_token_train = [char_dict[x.strip(' ').strip(', ').strip(' + ').strip("'")] for x in character_train]

In [ ]:
character_token_test = [char_dict[x.strip(' ').strip(', ').strip(' + ').strip("'")] for x in character_test]

In [ ]:
char_dict

In [ ]:
character_test[:5]

In [ ]:
character_token_test[:5]

Now you have got the tokenized datasets for both your dialogue and character

### Build the CNN

In [ ]:
# what is the maximum length of any dialogue in train
max(len(x) for x in dialogue_train)

In [ ]:
max(len(x) for x in dialogue_test)

In [ ]:
# CNN wants all inputs to be of same length so we will use padding to the max length
from tensorflow.keras.preprocessing.sequence import pad_sequences
maxlen = 69
d_train = pad_sequences(dialogue_token_train, maxlen=maxlen, padding='post')
d_test = pad_sequences(dialogue_token_test, maxlen=maxlen, padding='post')

In [ ]:
from tensorflow.keras.layers import Dense, Embedding, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.models import Sequential
cnn = Sequential() # define the model first
cnn.add(Embedding(big_vocab_len, 8, input_length=maxlen)) # add embedding layer or input layer
cnn.add(Conv1D(128, 5, activation='relu')) # second conv layer
cnn.add(GlobalMaxPooling1D()) # downsample using Max pooling
cnn.add(Dense(128, activation='relu')) # first dense NN layer
cnn.add(Dense(3, activation='softmax')) # output
# compile to get the model from the network
cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print(cnn.summary())

#### Start training

In [ ]:
character_token_train[:2]

In [ ]:
# fit the model
from tensorflow.keras.utils import to_categorical
cnn.fit(d_train, to_categorical(character_token_train, num_classes=3), epochs=50)

#### Run predictions

In [ ]:
# print the character predictions for a few of the dialogues
for i in range(5,10):
    print(dialogue_test.tolist()[i])
    print(character_test.tolist()[i])
preds = cnn.predict(d_test[5:10])
for pred in preds:
    pred_index = pred.tolist().index(max(pred)) # we need the index of the max prob class from the list to get the character name
    char = [c for c in char_dict if char_dict[c] == pred_index]
    print("Predicted character for the dialogue is: " + str(char))

## Natural language generation with LSTM

### Create train and test datasets

In [17]:
from keras.preprocessing.text import Tokenizer
# for NLG we will create the training dataset with a set of words in input and the next word as the label
short_train = dialogue_train[:250]

Using TensorFlow backend.


In [64]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(short_train)
vocab_words = len(tokenizer.word_index) + 1

In [85]:
train_words = []
j = 0
for dialogue in short_train:
    j += 1
    word_list = tokenizer.texts_to_sequences([dialogue])[0]
    for i in range(1, len(word_list)):
        word_sequence = word_list[:i+1]
        train_words.append(word_sequence)
        if j == 1:
            print(tokenizer.sequences_to_texts([word_sequence]))

['yamato this']
['yamato this is']
['yamato this is the']
['yamato this is the enterprise']
['yamato this is the enterprise yamato']
['yamato this is the enterprise yamato come']
['yamato this is the enterprise yamato come in']


In [66]:
# Pad tokens to be of equal length
from tensorflow.keras.preprocessing.sequence import pad_sequences
train_word_tokens_padded = np.array(pad_sequences(train_words, maxlen=10, padding='pre'))
train_word_tokens = train_word_tokens_padded[:,:-1]
train_label_tokens = train_word_tokens_padded[:,-1]

In [67]:
label.shape

(1682,)

In [68]:
from tensorflow.keras.utils import to_categorical
y = to_categorical(train_label_tokens, num_classes=vocab_words)

### Train the LSTM model

In [88]:
from tensorflow.keras.layers import Dense, Dropout, LSTM, Embedding
from tensorflow.keras.models import Sequential
model = Sequential()
model.add(Embedding(vocab_words, 1, input_length=9))
model.add(LSTM(256))
model.add(Dropout(0.2))
model.add(Dense(vocab_words, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam')
model.summary()

Model: "sequential_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_2 (Embedding)      (None, 9, 1)              673       
_________________________________________________________________
lstm_2 (LSTM)                (None, 256)               264192    
_________________________________________________________________
dropout_2 (Dropout)          (None, 256)               0         
_________________________________________________________________
dense_2 (Dense)              (None, 673)               172961    
Total params: 437,826
Trainable params: 437,826
Non-trainable params: 0
_________________________________________________________________


In [89]:
model.fit(train_word_tokens, y, epochs=100)

Train on 1682 samples
Epoch 1/100
1682/1682 [==============================] - 1s 805us/sample - loss: 6.2185
Epoch 2/100
1682/1682 [==============================] - 0s 114us/sample - loss: 5.8370
Epoch 3/100
1682/1682 [==============================] - 0s 109us/sample - loss: 5.7612
Epoch 4/100
1682/1682 [==============================] - 0s 105us/sample - loss: 5.7479
Epoch 5/100
1682/1682 [==============================] - 0s 104us/sample - loss: 5.7463
Epoch 6/100
1682/1682 [==============================] - 0s 104us/sample - loss: 5.7296
Epoch 7/100
1682/1682 [==============================] - 0s 103us/sample - loss: 5.7190
Epoch 8/100
1682/1682 [==============================] - 0s 104us/sample - loss: 5.7183
Epoch 9/100
1682/1682 [==============================] - 0s 104us/sample - loss: 5.7072
Epoch 10/100
1682/1682 [==============================] - 0s 104us/sample - loss: 5.6906
Epoch 11/100
1682/1682 [==============================] - 0s 104us/sample - loss: 5.6823
Epoch 12

In [77]:
from numpy import random
begin = random.randint(len(train_word_tokens_padded))
try_1 = [train_word_tokens_padded[begin]]
try1_padded = pad_sequences(try_1, maxlen=9, padding='pre')
# predictions
pred = model.predict(try1_padded)
res = np.argmax(pred)
print("Input is: " + str(tokenizer.sequences_to_texts(try1_padded)))
print("Next word generated by model is: " + str(tokenizer.sequences_to_texts([[res]])))

Input is: ['commander william riker of the starship']
Next word generated by model is: ['enterprise']
